In [ ]:
!pip install transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoProcessor, 
    Qwen2VLForConditionalGeneration, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ==========================================
# 1. 초기화 및 GPU 메모리 정리
# ==========================================
torch.cuda.empty_cache()
model_id = "Qwen/Qwen2-VL-2B-Instruct"

print("⏳ Qwen2-VL 모델과 프로세서를 로드하는 중입니다...")

# ==========================================
# 2. 5060Ti 맞춤형 4-bit 양자화 (QLoRA) 설정
# ==========================================
# 모델을 4-bit로 압축하여 로드 (VRAM 사용량을 극적으로 줄여줌)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 모델 로드 (학습용)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 프로세서 로드 (텍스트와 이미지를 모델이 이해할 수 있게 변환하는 도구)
processor = AutoProcessor.from_pretrained(model_id)

# k-bit 학습을 위한 모델 준비
model = prepare_model_for_kbit_training(model)

# ==========================================
# 3. LoRA 어댑터 설정 (뇌의 일부분만 학습)
# ==========================================
# 전체 모델을 다 학습하면 5060Ti가 터집니다. 
# LoRA를 사용해 핵심 주의집중(Attention) 모듈만 콕 집어서 가볍게 학습시킵니다.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # 핵심 모듈만 타겟팅
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # 학습할 파라미터 비율 출력 (보통 1~2% 내외)

# ==========================================
# 4. 데이터셋 로드 (미리 만들어둔 1회성 정답지)
# ==========================================
# Phase 2에서 만들어둔 JSONL 파일을 불러옵니다.
dataset = load_dataset('json', data_files={'train': 'train_vlm_resized.jsonl'})

# ==========================================
# 5. SFTTrainer 학습 설정 및 시작
# ==========================================
training_args = TrainingArguments(
    output_dir="VQA_Qwen_LoRA",       # 학습된 가중치가 저장될 폴더
    num_train_epochs=3,               # VLM은 3~5번만 봐도 금방 똑똑해집니다.
    per_device_train_batch_size=2,    # 5060Ti VRAM 방어를 위해 작게 설정 (OOM 나면 1로 줄이세요)
    gradient_accumulation_steps=4,    # 배치 사이즈가 작아도 크게 학습하는 효과를 줌
    optim="paged_adamw_8bit",         # 메모리 절약형 옵티마이저
    save_strategy="epoch",            # 매 에포크마다 저장
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,                        # 속도 향상을 위한 Mixed Precision
    max_grad_norm=0.3,
    warmup_ratio=0.03,
)

# 트레이너 객체 생성
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    peft_config=lora_config,
    # Qwen2-VL이 요구하는 특수 데이터 패킹 처리
    dataset_text_field="messages", 
)

print("\n🔥 Qwen-VL 학습을 시작합니다! (GPU 팬이 강하게 돌 수 있습니다)")

if __name__ == '__main__':
    # 학습 시작
    trainer.train()
    
    # 학습이 끝난 후 최종 LoRA 가중치를 안전하게 디스크에 저장
    trainer.model.save_pretrained("VQA_Qwen_LoRA/final_model")
    processor.save_pretrained("VQA_Qwen_LoRA/final_model")
    print("\n🎉 VLM 학습이 완료되었습니다! 최종 모델이 'VQA_Qwen_LoRA/final_model'에 저장되었습니다.")